[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VectorInstitute/synthetic-data-bootcamp/blob/main/implementations/qa_text_generation/03_quality_filtering.ipynb)

# Step 3 — Quality Filtering and LLM-as-Judge

Filter synthetic Q&A with cheap heuristics, then score survivors with a judge model.

## Learning objectives
- Apply deduplication, format checks, and prompt-leakage detection
- Score samples on correctness, coherence, instruction-following, and plausibility
- Optionally compare candidates pairwise against seed examples

In [1]:
import os
from pathlib import Path

from aieng.syn_data.text import (
    DEFAULT_JUDGE_THRESHOLD,
    RESULTS_DIR,
    SYNTHETIC_FILTERED_PATH,
    SYNTHETIC_RAW_PATH,
    QASample,
    apply_heuristic_filters,
    create_judge_client,
    filter_with_judge,
    load_implementation_dotenv,
    load_typed_jsonl,
    save_typed_jsonl,
    summarize_heuristic_rejections,
    summarize_judge_scores,
    use_repo_root,
    write_json,
)
from rich import box
from rich.console import Console
from rich.table import Table


# Setting the notebook directory to the project's root folder
if Path("").absolute().name == "synthetic-data-bootcamp":
    print(f"Notebook path is already the root path: {Path('').absolute()}")
else:
    os.chdir(Path("").absolute().parent.parent)
    print(f"The notebook path has been set to: {Path('').absolute()}")

load_implementation_dotenv()
use_repo_root(Path("."))

console = Console(width=100)

The notebook path has been set to: /home/coder/synthetic-data-bootcamp


In [2]:
# TODO: remove this before merging into main
%load_ext autoreload
%autoreload 2

## 1. Heuristic filtering

In [4]:
raw_samples = load_typed_jsonl(SYNTHETIC_RAW_PATH, QASample.from_dict)
kept, rejected = apply_heuristic_filters(raw_samples)
print(f"Kept {len(kept)} / {len(raw_samples)} samples")

reason_counts = summarize_heuristic_rejections(rejected)
print("Rejection reasons:", reason_counts)

# Inspect a few rejected samples and which heuristic(s) fired
table = Table(title="Sample heuristic rejections", show_lines=True)
table.add_column("id", style="cyan", max_width=20)
table.add_column("reason(s)", style="red")
table.add_column("question", overflow="fold")
for row in rejected[:8]:
    table.add_row(row.get("id", ""), row.get("reasons", row.get("reason", "")), row.get("question", ""))
console.print(table)

Kept 135 / 144 samples
Rejection reasons: {'duplicate_question': 9}


                                    Sample heuristic rejections                                     
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ id                   ┃ reason(s)          ┃ question                                             ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ d9c3cd28-0bdb-47a1-… │ duplicate_question │ Under the 'PROMISE TO PAY' section, what are the     │
│                      │                    │ specific types of charges that you promise to pay if │
│                      │                    │ they are made to y                                   │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ f2ff0759-ce0a-44dd-… │ duplicate_question │ Under what specific condition can a Statement Copy   │
│                      │                    │ Fee be charged to an account, and what is the        │
│                      │                    │ explicit exception to t                              │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 5d307b20-977b-47ed-… │ duplicate_question │ Under what condition can a change made to the        │
│                      │                    │ Consumer Credit Card Agreement by the Credit Union   │
│                      │                    │ apply to your existing                               │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ c4c7450a-d7d8-45d8-… │ duplicate_question │ Under what specific conditions will the issuer close │
│                      │                    │ a primary account holder's account and require them  │
│                      │                    │ to apply for a                                       │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 4b692639-0b81-4a79-… │ duplicate_question │ Under the 'NO WAIVER' provision, what is the         │
│                      │                    │ consequence if the Credit Union repeatedly delays    │
│                      │                    │ enforcing its rights?                                │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 2e8be074-b274-447d-… │ duplicate_question │ If a customer suspects an error and reports it via a │
│                      │                    │ phone call, what are the two consequences regarding  │
│                      │                    │ the investigati                                      │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ c5704cbf-d453-400c-… │ duplicate_question │ According to the Updated Investor Bulletin from      │
│                      │                    │ April 23, 2026, which specific presidential          │
│                      │                    │ directive prompted the SEC's                         │
├──────────────────────┼────────────────────┼──────────────────────────────────────────────────────┤
│ 9d8ecc06-d25f-4362-… │ duplicate_question │ According to the passage, what are the specific      │
│                      │                    │ criteria that a passphrase must meet to be           │
│                      │                    │ considered "strong," and what                        │
└──────────────────────┴────────────────────┴──────────────────────────────────────────────────────┘

## 2. LLM-as-judge absolute scoring

Here the judge model evaluates **synthetic Q&A quality** (question + gold answer vs the source passage). This is different from notebooks 01/05, where the judge scores a **model response** against a reference answer after inference.

The minimum pass score is defined in configs (`DEFAULT_JUDGE_THRESHOLD`).


In [5]:
# TODO: change judge model to another family of models (e.g claude or gpt-4o)
judge = create_judge_client()

heuristic_rejected = rejected
filtered_samples, judge_scores, judge_rejected = filter_with_judge(
    judge,
    kept,
    threshold=DEFAULT_JUDGE_THRESHOLD,
)
# filter_with_judge re-runs heuristics on `kept`, so keep the original heuristic
# rejections and append judge-only rejects for an accurate quality report.
rejected = heuristic_rejected + [
    row for row in judge_rejected if row.get("reason") == "below_judge_threshold"
]
console.print(
    f"[bold green]After judge filter:[/bold green] [yellow]{len(filtered_samples)}[/yellow] kept, "
    f"[red]{len(rejected)}[/red] rejected "
    f"[dim]({len(heuristic_rejected)} heuristic + "
    f"{len(rejected) - len(heuristic_rejected)} judge)[/dim]"
)
summarize_judge_scores(judge_scores)


2026-07-28 22:19:49,513 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 783137e2-cc78-4f6d-b8ff-44db910ec979


2026-07-28 22:19:50,894 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The question is clear and directly addresses the text, and the gold answer is highly accurate and fully grounded in the provided passage."
*********** End of JSON payload ***********
2026-07-28 22:19:50,897 INFO aieng.syn_data.text.judge: Scoring synthetic Q&A quality for sample: 261b8810-8a7c-455b-8412-d27148124d3c
2026-07-28 22:19:52,042 DEBUG aieng.syn_data.text.clients: *********** Extracted JSON payload: *********** 
{
    "correctness": 5,
    "coherence": 5,
    "instruction_following": 5,
    "factual_plausibility": 5,
    "reasoning": "The question is highly specific and directly answered by the provided text. The gold answer is perfectly accurate and faithful to the passage."
}
*********** End of JSON payload ***********
2026-07-28 22:19:52,044 IN

After judge filter: 135 kept, 0 rejected (heuristics + judge)

{'correctness': 4.985185185185185,
 'coherence': 5.0,
 'instruction_following': 4.985185185185185,
 'factual_plausibility': 5.0,
 'average': 4.992592592592593}

## 3. Save filtered corpus and quality report

In [6]:
save_typed_jsonl(
    SYNTHETIC_FILTERED_PATH,
    filtered_samples,
    to_dict=QASample.to_dict,
)

quality_report = {
    "input_count": len(raw_samples),
    "after_heuristics": len(kept),
    "after_judge": len(filtered_samples),
    "judge_threshold": DEFAULT_JUDGE_THRESHOLD,
    "judge_summary": summarize_judge_scores(judge_scores),
    "heuristic_rejected_count": len(heuristic_rejected),
    "judge_rejected_count": len(rejected) - len(heuristic_rejected),
    "rejected": rejected,
}
write_json(RESULTS_DIR / "quality_report.json", quality_report)


table = Table(title="Quality Report", box=box.ROUNDED)
table.add_column("Metric", style="bold cyan")
table.add_column("Value", style="bold yellow")

for k, v in quality_report.items():
    if isinstance(v, dict):
        # If value is a dictionary, show sub-keys and values
        for subk, subv in v.items():
            table.add_row(f"{k}.{subk}", str(subv))
    elif isinstance(v, list):
        table.add_row(k, f"{len(v)} items")
    else:
        table.add_row(k, str(v))
console.print(table)


                      Quality Report                       
╭─────────────────────────────────────┬───────────────────╮
│ Metric                              │ Value             │
├─────────────────────────────────────┼───────────────────┤
│ input_count                         │ 144               │
│ after_heuristics                    │ 135               │
│ after_judge                         │ 135               │
│ judge_threshold                     │ 3.5               │
│ judge_summary.correctness           │ 4.985185185185185 │
│ judge_summary.coherence             │ 5.0               │
│ judge_summary.instruction_following │ 4.985185185185185 │
│ judge_summary.factual_plausibility  │ 5.0               │
│ judge_summary.average               │ 4.992592592592593 │
│ rejected                            │ 0 items           │
╰─────────────────────────────────────┴───────────────────╯